In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import random
import numpy as np
import copy
import matplotlib.pyplot as plt


import os
os.environ["CUDA_VISIBLE_DEVICES"]="1"

In [3]:
# Set device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [14]:
from torch.utils.cpp_extension import load
# Load and compile CUDA extension
maxplus_conv2d = load(
    name="maxplus_conv2d",
    sources=["conv2d_cuda.cu"],
    # extra_cuda_cflags=["-lineinfo"],  # Optional: Debugging info
    verbose=True
)

from torch.autograd import Function    
class MaxPlusConv2dFunction(Function):
    @staticmethod
    def forward(ctx, input, weight, bias, stride, padding):
        output, argmax_input_idx, argmax_weight_idx = maxplus_conv2d.maxplus_conv2d_forward(
            input, weight, bias, stride, padding
        )
        ctx.save_for_backward(argmax_input_idx, argmax_weight_idx)
        ctx.input_shape = input.shape
        ctx.weight_shape = weight.shape
        ctx.stride = stride
        ctx.padding = padding
        return output

    @staticmethod
    def backward(ctx, grad_output):
        argmax_input_idx, argmax_weight_idx = ctx.saved_tensors
        B, C_in, H_in, W_in = ctx.input_shape
        C_out, _, K, _ = ctx.weight_shape
        stride = ctx.stride
        padding = ctx.padding
        H_out = (H_in + 2 * padding - K) // stride + 1
        W_out = (W_in + 2 * padding - K) // stride + 1

        grad_input, grad_weight, grad_bias = maxplus_conv2d.maxplus_conv2d_backward(
            grad_output, argmax_input_idx, argmax_weight_idx,
            B, C_in, C_out, H_in, W_in, H_out, W_out, K
        )
        return grad_input, grad_weight, grad_bias, None, None

# Convenience wrapper
def maxplus_conv2d_wrapper(input, weight, bias=None, stride=1, padding=0):
    return MaxPlusConv2dFunction.apply(input, weight, bias, stride, padding)

class MorphConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=(3,3), stride=(1,1), padding=(1,1), bias=True, alpha=1.0):
        super(MorphConv2d, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.bias = bias
        self.alpha = alpha

        self.weight = nn.Parameter(
            torch.normal(mean=0.0, std=alpha, size=(out_channels, in_channels, kernel_size[0], kernel_size[1]))
        )
        if bias:
            self.b = nn.Parameter(torch.normal(mean=0.0, std=1.0, size=(out_channels, )))
            self.b2 = nn.Parameter(torch.normal(mean=0.0, std=1.0, size=(out_channels, )))
        else:
            self.b = nn.Parameter(torch.zeros((out_channels, )) - 1e9)
            self.b2 = nn.Parameter(torch.zeros((out_channels, )) + 1e9)

    def forward(self, x):
        # Max morphological operation
        max_w = self.weight
        x_max = maxplus_conv2d_wrapper(x, max_w, self.b, self.stride[0], self.padding)

        # Min morphological operation
        min_w = self.weight
        x_min = -x
        x_min = maxplus_conv2d_wrapper(x_min, -min_w, -self.b2, self.stride[0], self.padding)
        x_min = -x_min

        # Aggregation of max and min operations
        x_out = (x_max + x_min)/2

        return x_out
    
class ConvLinAct(nn.Module):
    def __init__(self, channels, method="simple"):
        super(ConvLinAct, self).__init__()
        self.method = method

        if method == 'simple':
            self.a = nn.Parameter(torch.zeros(3, 3, channels))
            self.a.data[1,1,:] += 1
        else:
            tmp = nn.Conv2d(channels, channels, 3, bias = False).weight.data.detach()
            tmp = torch.transpose(tmp, 0, 2)
            tmp = torch.transpose(tmp, 1, 3)
            U, S, Vh = np.linalg.svd(tmp.cpu().numpy(), full_matrices=True)
            self.U = torch.tensor(U).to(device)
            self.Vh = torch.tensor(Vh).to(device)
            self.a = torch.zeros(3, 3, channels).to(device)
            self.a[:, :, :S.shape[2]] = torch.tensor(S)
            self.a = nn.Parameter(self.a)
    def forward(self, x):
        if self.method == 'simple':
            # x = x * self.a.unsqueeze(-1).unsqueeze(-1).unsqueeze(0).repeat(x.size(0), 1, x.size(2), x.size(3))
            tmp = torch.diag_embed(self.a)
            tmp = torch.transpose(tmp, 0, 2)
            tmp = torch.transpose(tmp, 1, 3)
            x = nn.functional.conv2d(x, tmp, padding=1)
        else:
            tmp = self.U @ (self.a.unsqueeze(-1) * self.Vh)
            tmp = torch.transpose(tmp, 0, 2)
            tmp = torch.transpose(tmp, 1, 3)
            x = nn.functional.conv2d(x, tmp, padding=1)
        return x

class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, kernel_size=1, stride=1, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNet20(nn.Module):
    def __init__(self, in_channels=3, num_blocks=[3, 3, 3], num_classes=10):
        super().__init__()
        self.in_planes = 16
        self.conv1 = nn.Conv2d(in_channels, 16, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)

        self.layer1 = self._make_layer(16, num_blocks[0])
        self.pool1 = nn.MaxPool2d(2, 2)

        self.layer2 = self._make_layer(32, num_blocks[1])
        self.pool2 = nn.MaxPool2d(2, 2)

        self.layer3 = self._make_layer(64, num_blocks[2])
        
        self.fc = nn.Linear(64, num_classes)

    def _make_layer(self, planes, num_blocks):
        layers = []
        for _ in range(num_blocks):
            layers.append(BasicBlock(self.in_planes, planes))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))

        out = self.layer1(out)
        out = self.pool1(out)

        out = self.layer2(out)
        out = self.pool2(out)

        out = self.layer3(out)
        
        out = F.avg_pool2d(out, out.shape[2]) 
        out = out.view(out.size(0), -1)
        return self.fc(out)  


class MPM_BasicBlock(nn.Module):
    def __init__(self, in_planes, planes):
        super().__init__()
        self.conv1 = MorphConv2d(in_planes, planes, kernel_size=(3,3), padding=1, bias=False)
        self.linact1 = ConvLinAct(planes, method="simple")
        # self.bn1 = nn.BatchNorm2d(planes)
        self.bn1 = nn.Identity()
        self.conv2 = MorphConv2d(planes, planes, kernel_size=(3,3), padding=1, bias=False)
        self.linact2 = ConvLinAct(planes, method="simple")
        # self.bn2 = nn.BatchNorm2d(planes)
        self.bn2 = nn.Identity()

        self.shortcut = nn.Sequential()
        if in_planes != planes:
            self.shortcut = MorphConv2d(in_planes, planes, kernel_size=(1,1), padding=0, bias=False)

    def forward(self, x):
        out = self.linact1(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.linact2(out)
        return out

class MPM_ResNet20(nn.Module):
    def __init__(self, in_channels=3, num_blocks=[3,3,3], num_classes=10):
        super().__init__()
        self.in_planes = 16

        self.conv1 = MorphConv2d(in_channels, 16, kernel_size=(3, 3), padding=1, bias=False)
        self.linact1 = ConvLinAct(16, method = "simple")
        # self.bn1 = nn.BatchNorm2d(16)
        self.bn1 = nn.Identity()

        self.layer1 = self._make_layer(16, num_blocks[0])
        self.pool1 = nn.MaxPool2d(2, 2)

        self.layer2 = self._make_layer(32, num_blocks[1])
        self.pool2 = nn.MaxPool2d(2, 2)

        self.layer3 = self._make_layer(64, num_blocks[2])
        
        self.fc = nn.Linear(64, num_classes)

    def _make_layer(self, planes, num_blocks):
        layers = []
        for _ in range(num_blocks):
            layers.append(MPM_BasicBlock(self.in_planes, planes))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.linact1(self.bn1(self.conv1(x)))

        out = self.layer1(out)
        out = self.pool1(out)

        out = self.layer2(out)
        out = self.pool2(out)

        out = self.layer3(out)
        
        out = F.avg_pool2d(out, out.shape[2]) 
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out  

Using /home/anonymous/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module maxplus_conv2d, skipping build step...
Loading extension module maxplus_conv2d...


In [6]:
def train(model, criterion, optimizer, train_loader, val_loader, num_epochs=50, return_list=False):
    # Training and validation loop
    best_val_accuracy = 0.0
    best_model = None

    train_list = []
    val_list = []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Validation phase
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        train_accuracy = 100 * correct / total
        train_list.append(train_accuracy)
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_accuracy = 100 * correct / total
        val_list.append(val_accuracy)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Train Accuracy: {train_accuracy:.2f}%, Validation Accuracy: {val_accuracy:.2f}%")

        # Save best model based on validation accuracy
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model = copy.deepcopy(model)

    if return_list:
        return best_model, train_list, val_list
    else:
        return best_model

def test(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy on the test set: {accuracy:.2f}%')

In [7]:
# Load and preprocess MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))])

full_train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

# Split train dataset into training and validation sets
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

Files already downloaded and verified
Files already downloaded and verified


In [ ]:
# Initialize model, loss function, and optimizer
model = ResNet20(in_channels=3).to(device)

total_params = 0
for param in model.parameters():
    total_params += param.numel()
print(f"Total number of parameters: {total_params}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 272282
Epoch [1/50], Loss: 0.9633, Train Accuracy: 55.66%, Validation Accuracy: 55.32%
Epoch [2/50], Loss: 0.7398, Train Accuracy: 71.56%, Validation Accuracy: 69.51%
Epoch [3/50], Loss: 0.5540, Train Accuracy: 76.03%, Validation Accuracy: 73.47%
Epoch [4/50], Loss: 0.5991, Train Accuracy: 77.12%, Validation Accuracy: 73.53%
Epoch [5/50], Loss: 0.5236, Train Accuracy: 75.00%, Validation Accuracy: 71.33%
Epoch [6/50], Loss: 0.7248, Train Accuracy: 80.88%, Validation Accuracy: 75.59%
Epoch [7/50], Loss: 0.6187, Train Accuracy: 83.93%, Validation Accuracy: 77.67%
Epoch [8/50], Loss: 0.2934, Train Accuracy: 84.95%, Validation Accuracy: 77.87%
Epoch [9/50], Loss: 0.5020, Train Accuracy: 86.96%, Validation Accuracy: 78.99%
Epoch [10/50], Loss: 0.3547, Train Accuracy: 90.15%, Validation Accuracy: 80.86%
Epoch [11/50], Loss: 0.3259, Train Accuracy: 91.32%, Validation Accuracy: 81.01%
Epoch [12/50], Loss: 0.1717, Train Accuracy: 92.48%, Validation Accuracy: 81.43%
Ep

In [9]:
test(model, test_loader)

Accuracy on the test set: 81.78%


In [ ]:
# Initialize model, loss function, and optimizer
model = MPM_ResNet20(in_channels=3).to(device)

total_params = 0
for param in model.parameters():
    total_params += param.numel()
print(f"Total number of parameters: {total_params}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader, num_epochs=100)

Total number of parameters: 278666
Epoch [1/100], Loss: 2.2722, Train Accuracy: 24.02%, Validation Accuracy: 23.37%
Epoch [2/100], Loss: 2.1399, Train Accuracy: 24.91%, Validation Accuracy: 24.41%
Epoch [3/100], Loss: 1.9547, Train Accuracy: 27.80%, Validation Accuracy: 27.40%
Epoch [4/100], Loss: 1.8308, Train Accuracy: 30.87%, Validation Accuracy: 31.14%
Epoch [5/100], Loss: 1.8554, Train Accuracy: 33.47%, Validation Accuracy: 32.50%
Epoch [6/100], Loss: 1.6573, Train Accuracy: 35.53%, Validation Accuracy: 34.39%
Epoch [7/100], Loss: 1.7063, Train Accuracy: 38.81%, Validation Accuracy: 38.24%
Epoch [8/100], Loss: 1.5504, Train Accuracy: 41.00%, Validation Accuracy: 40.45%
Epoch [9/100], Loss: 1.5288, Train Accuracy: 41.97%, Validation Accuracy: 41.39%
Epoch [10/100], Loss: 1.5041, Train Accuracy: 44.95%, Validation Accuracy: 44.76%
Epoch [11/100], Loss: 1.4358, Train Accuracy: 48.33%, Validation Accuracy: 47.68%
Epoch [12/100], Loss: 1.2631, Train Accuracy: 49.78%, Validation Accurac

In [11]:
test(model, test_loader)

Accuracy on the test set: 62.46%


In [12]:
# Load and preprocess MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

full_train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.FashionMNIST(root='./data', train=False, transform=transform, download=True)

# Split train dataset into training and validation sets
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [15]:
# Initialize model, loss function, and optimizer
model = ResNet20(in_channels=1).to(device)

total_params = 0
for param in model.parameters():
    total_params += param.numel()
print(f"Total number of parameters: {total_params}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 271994
Epoch [1/50], Loss: 0.1964, Train Accuracy: 87.08%, Validation Accuracy: 86.48%
Epoch [2/50], Loss: 0.1522, Train Accuracy: 91.65%, Validation Accuracy: 90.17%
Epoch [3/50], Loss: 0.1310, Train Accuracy: 89.87%, Validation Accuracy: 88.56%
Epoch [4/50], Loss: 0.2337, Train Accuracy: 93.36%, Validation Accuracy: 91.54%
Epoch [5/50], Loss: 0.1576, Train Accuracy: 93.58%, Validation Accuracy: 91.73%
Epoch [6/50], Loss: 0.1549, Train Accuracy: 92.24%, Validation Accuracy: 89.82%
Epoch [7/50], Loss: 0.0627, Train Accuracy: 94.81%, Validation Accuracy: 91.85%
Epoch [8/50], Loss: 0.0654, Train Accuracy: 94.05%, Validation Accuracy: 90.87%
Epoch [9/50], Loss: 0.1364, Train Accuracy: 96.05%, Validation Accuracy: 92.52%
Epoch [10/50], Loss: 0.1200, Train Accuracy: 96.64%, Validation Accuracy: 92.53%
Epoch [11/50], Loss: 0.1067, Train Accuracy: 95.38%, Validation Accuracy: 91.46%
Epoch [12/50], Loss: 0.0822, Train Accuracy: 95.79%, Validation Accuracy: 91.37%
Ep

In [16]:
test(model, test_loader)

Accuracy on the test set: 92.37%


In [17]:
# Initialize model, loss function, and optimizer
model = MPM_ResNet20(in_channels=1).to(device)

total_params = 0
for param in model.parameters():
    total_params += param.numel()
print(f"Total number of parameters: {total_params}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = train(model, criterion, optimizer, train_loader, val_loader)

Total number of parameters: 278378
Epoch [1/50], Loss: 1.1933, Train Accuracy: 63.71%, Validation Accuracy: 63.23%
Epoch [2/50], Loss: 0.6589, Train Accuracy: 71.49%, Validation Accuracy: 71.04%
Epoch [3/50], Loss: 0.5494, Train Accuracy: 73.71%, Validation Accuracy: 73.35%
Epoch [4/50], Loss: 0.8381, Train Accuracy: 75.20%, Validation Accuracy: 74.78%
Epoch [5/50], Loss: 0.5248, Train Accuracy: 79.35%, Validation Accuracy: 79.12%
Epoch [6/50], Loss: 0.5997, Train Accuracy: 80.57%, Validation Accuracy: 80.58%
Epoch [7/50], Loss: 0.5012, Train Accuracy: 82.39%, Validation Accuracy: 82.68%
Epoch [8/50], Loss: 0.4954, Train Accuracy: 85.02%, Validation Accuracy: 84.35%
Epoch [9/50], Loss: 0.3508, Train Accuracy: 85.90%, Validation Accuracy: 85.55%
Epoch [10/50], Loss: 0.3263, Train Accuracy: 85.63%, Validation Accuracy: 85.25%
Epoch [11/50], Loss: 0.4827, Train Accuracy: 87.42%, Validation Accuracy: 86.47%
Epoch [12/50], Loss: 0.3757, Train Accuracy: 87.59%, Validation Accuracy: 86.55%
Ep

In [18]:
test(model, test_loader)

Accuracy on the test set: 89.64%
